This notebooks contains a very rough benchmark for 'jina-clip-v2' on my CPU (Apple M2):
- [Done] Original PyTorch implementation from https://huggingface.co/jinaai/jina-clip-v2
- [TODO] Quantized versions (int8/fp16)
- [TODO] Quantized versions converted to ONNX and run with ONNX Runtime

In [7]:
%pip install transformers torch torchvision requests Pillow --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


In [1]:
import io
import statistics
import time

import requests
import torch
from PIL import Image
from transformers import AutoModel

/opt/miniconda3/envs/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Auxiliary functions

In [2]:
def measure(fn: callable, warmup: int, runs: int) -> dict:
    """Measure the execution time of a function

    Args:
        fn (callable): The function to measure
        warmup (int): Number of warmup runs to perform before measuring
        runs (int): Number of runs to perform for measuring

    Returns:
        dict: A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times in milliseconds
    """
    
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)

    return {
        "mean":   statistics.mean(times),
        "median": statistics.median(times),
        "stdev":  statistics.stdev(times) if len(times) > 1 else 0.0,
        "min":    min(times),
        "max":    max(times),
    }

def print_table(label: str, text_s: dict, image_s: dict) -> None:
    """Print a formatted table comparing text and image execution times

    Args:
        label (str): The label for the table
        text_s (dict): A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times for text processing in milliseconds
        image_s (dict): A dictionary containing the mean, median, standard deviation, minimum, and maximum execution times for image processing in milliseconds
    """

    print(f"\n{'─' * 57}")
    print(label)
    print(f"{'─' * 57}")
    print(f"{'':22} {'Text':>12} {'Image':>12}")
    for k, name in [("mean", "Average"), ("median", "Median"), ("stdev", "Std Dev"), ("min", "Minimum"), ("max", "Maximum")]:
        print(f"{name:22} {text_s[k]:>10.1f} ms {image_s[k]:>10.1f} ms")

## Text/image examples

In [3]:
TEST_TEXT = "brown fox jumps over the lazy dog"
TEST_IMAGE_URL = "https://4.img-dpreview.com/files/p/E~C667x0S5333x4000T1200x900~articles/3925134721/0266554465.jpeg"

response = requests.get(TEST_IMAGE_URL)
if response.status_code != 200:
    raise Exception(f"Failed to download image from {TEST_IMAGE_URL}, code: {response.status_code}")
TEST_IMAGE = Image.open(io.BytesIO(response.content)).convert("RGB")

## Benchmark params

In [4]:
WARMUP = 3
RUNS = 20

## Model metdata

In [5]:
MODEL_ID = "jinaai/jina-clip-v2"

## PyTorch benchmark

In [6]:
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()

with torch.no_grad():
    text_s = measure(
        lambda: model.encode_text([TEST_TEXT]),
        WARMUP,
        RUNS
    )
    image_s = measure(
        lambda: model.encode_image([TEST_IMAGE]),
        WARMUP, 
        RUNS
    )
print_table("PyTorch float32 (CPU)", text_s, image_s)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
/Users/alex/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:137: UserWarning: Flash attention requires CUDA, disabling
  warnings.warn('Flash attention requires CUDA, disabling')
/Users/alex/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:172: UserWarning: xFormers requires CUDA, disabling
  warnings.warn('xFormers requires CUDA, disabling')
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



─────────────────────────────────────────────────────────
PyTorch float32 (CPU)
─────────────────────────────────────────────────────────
                               Text        Image
Average                     583.7 ms     2033.6 ms
Median                      579.9 ms     2030.4 ms
Std Dev                      18.7 ms       14.0 ms
Minimum                     563.9 ms     2019.7 ms
Maximum                     631.7 ms     2081.6 ms
